# W08 影像前處理與自動化流程

**對合成影像做重取樣，比較遮罩體素數；確認label=1才是結節。**

上課 2026/10/22（四）。只改ROOT。不要上傳可識別病人資料。


In [ ]:
# 這段在做什麼：掛上 Drive、設定 ROOT，並做環境自檢。
# 下面會印出套件版本與 ROOT 是否存在。跑不過是環境的問題，不是你的問題——
# 把整行錯誤訊息貼進題本的「卡住的行號／我還不知道的」欄，判斷題照常作答、照常計分。
import sys, platform
from pathlib import Path
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/ndmu-radiomics-ai')
except ImportError:
    ROOT = Path.cwd()
print('Python  ', platform.python_version())
for _m in ('numpy', 'pandas'):
    try:
        print(f'{_m:8s}', __import__(_m).__version__)
    except ImportError:
        print(f'{_m:8s} 未安裝（本週用得到就要裝）')
print('ROOT    ', ROOT, '｜存在：', ROOT.exists())
(ROOT / 'results').mkdir(parents=True, exist_ok=True)


In [ ]:
# 這段在做什麼：固定亂數種子，讓同一份輸入每次跑出同樣的結果。
import numpy as np
import pandas as pd
np.random.seed(42)
print('seed=42')


## STEP 0　基準線

把印出來的三個 GLCM 數字抄到紙的最上面一行。這就是 `pipeline_00.py` 做的事。


In [ ]:
# 這段在做什麼：跑 pipeline_00 的內容——什麼前處理都不做，先量出基準線。
import sys
sys.path.append(str(ROOT / 'notebooks' / 'W08'))   # _common.py 在這裡
from _common import phantom, resample, report, SPACING_TARGET

vol, msk, sp = phantom()
base = report('STEP 0　基準線（原始非等向體素，無前處理）', vol, msk, sp)


## LAYER ①　重取樣到 1×1×1 mm

影像用線性內插、遮罩用最近鄰。這就是 `pipeline_01.py` 做的事。


In [ ]:
# 這段在做什麼：跑 pipeline_01 的內容——重取樣成等向體素，再量一次同樣三個數字。
vol_r = resample(vol, sp, SPACING_TARGET, order=1)                        # 影像：線性
msk_r = (resample(msk.astype(float), sp, SPACING_TARGET, order=0) > 0.5).astype(int)  # 遮罩：最近鄰
new = report('LAYER ①　重取樣到 1×1×1 mm', vol_r, msk_r, SPACING_TARGET)


In [ ]:
# 這段在做什麼：把前後兩組數字並排。A1 要的就是這張表上的數字。
print('--- 比對（A1 抄這裡）---')
for k, lab in [('contrast','GLCM 對比度'), ('homogeneity','GLCM 同質性'), ('energy','GLCM 能量')]:
    dlt = (new[k] - base[k]) / base[k] * 100
    print(f'  {lab:<12} {base[k]:>10.4f} → {new[k]:>10.4f}   變化 {dlt:+7.1f}%')
print(f"  {'ROI 體素數':<12} {base['n']:>10} → {new['n']:>10}   變化 {(new['n']-base['n'])/base['n']*100:+7.1f}%")
print(f"  {'ROI 體積(mm3)':<12} {base['vol']:>10.1f} → {new['vol']:>10.1f}   變化 {(new['vol']-base['vol'])/base['vol']*100:+7.1f}%")
print()
print('體素數變多是因為格子變密，不是腫瘤變大。這兩件事要分清楚。')


## 繳交

把上面「比對」那張表的數字抄進題本 A1：重取樣前後的尺寸、遮罩體素數、ROI 體積。
卡住就把整行錯誤訊息貼進題本的「卡住的行號／我還不知道的」欄——判斷題照常作答、照常計分。

課堂只跑這兩層；第 ③–⑤ 層（濾波、GLCM 階數、特徵表）是示範，不用動手。
